In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedGroupKFold
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
_csv = next(
    (p for p in (Path("predictor/data/dataset1.csv"), Path("../predictor/data/dataset1.csv")) if p.is_file()),
    None,
)
if _csv is None:
    raise FileNotFoundError("dataset1.csv not found; run dataset.ipynb from customer_prediction_system or adjust paths.")
df = pd.read_csv(_csv)

# Drop same-day rows (likelihood=0 is noise)
df = df[df['likelihood_prediction'] > 0].copy()

# Print exactly what columns came in
print("Columns in CSV:", list(df.columns))
print("Total columns:", len(df.columns))

print("Shape after dropping same-day rows:", df.shape)
print("likelihood_prediction stats:")
print(df['likelihood_prediction'].describe())

# Balanced 3-class target using quantiles so each class has ~equal rows
q33 = df['likelihood_prediction'].quantile(0.33)
q66 = df['likelihood_prediction'].quantile(0.66)
print(f"\nBucket boundaries: 0-{q33:.0f} days | {q33:.0f}-{q66:.0f} days | {q66:.0f}+ days")

df['timing_category'] = pd.cut(
    df['likelihood_prediction'],
    bins=[0, q33, q66, df['likelihood_prediction'].max() + 1],
    labels=[0, 1, 2]
).astype(int)

print("\nClass distribution:")
print(df['timing_category'].value_counts().sort_index())

# Define full feature list — any that exist in the CSV will be used
all_possible_features = [
    'amount', 'customer_transaction_count', 'customer_vendor_txn_count',
    'days_since_first_purchase', 'customer_avg_spend', 'customer_avg_gap',
    'days_since_last_transaction',
    'lag_1_gap', 'lag_2_gap', 'lag_3_gap',
    'rolling_avg_gap_3', 'rolling_avg_gap_7', 'rolling_avg_gap_14',
    'gap_trend_short', 'gap_trend_long', 'spend_trend',
    'rolling_spend_3', 'rolling_spend_7',
    'vendor_gap_vs_avg', 'vendor_spend_vs_avg', 'vendor_category_code',
    'transaction_month', 'transaction_weekday',
]

features = [f for f in all_possible_features if f in df.columns]
missing = [f for f in all_possible_features if f not in df.columns]

print(f"\nUsing {len(features)} features")
print("Missing from CSV (need to add in dataset.ipynb):", missing)

X = df[features].fillna(0)
y = df['timing_category']
groups = df['customer_id']

cv = StratifiedGroupKFold(n_splits=5)

# Single best model with tuned parameters
best_model = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    max_depth=10,
    min_samples_leaf=3,
    max_features='sqrt',
    random_state=42
)

scores = cross_val_score(best_model, X, y, groups=groups, cv=cv, scoring='f1_macro')
print(f"RandomForest (tuned): {scores.mean():.3f} ± {scores.std():.3f}")

best_model.fit(X, y)
print("\nClassification report:")
print(classification_report(y, best_model.predict(X), target_names=['soon','medium','later']))

print("\nFeature importance:")
imp = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False)
print(imp.round(4))

_models_dir = Path("predictor/models")
if not _models_dir.is_absolute() and not _models_dir.parent.exists():
    _models_dir = Path("../predictor/models")
_models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, _models_dir / "timing_model.pkl")
joblib.dump(features, _models_dir / "feature_list.pkl")
joblib.dump({'q33': float(q33), 'q66': float(q66)}, _models_dir / "bucket_boundaries.pkl")
print("\nModel saved.")


Columns in CSV: ['first_name', 'last_name', 'customer_id', 'transaction_id', 'transaction_datetime', 'amount', 'vendor', 'transaction_type', 'next_transaction_date', 'likelihood_prediction', 'customer_transaction_count', 'customer_vendor_txn_count', 'days_since_first_purchase', 'transaction_month', 'transaction_weekday', 'customer_avg_spend', 'customer_avg_gap', 'days_since_last_transaction', 'lag_1_gap', 'lag_2_gap', 'lag_3_gap', 'rolling_avg_gap_7', 'rolling_avg_gap_3', 'rolling_avg_gap_14', 'gap_trend_short', 'gap_trend_long', 'rolling_spend_3', 'rolling_spend_7', 'spend_trend', 'vendor_gap_vs_avg', 'vendor_spend_vs_avg', 'vendor_category', 'vendor_category_code', 'transaction_day', 'vendor_preferred_day', 'vendor_day_std', 'vendor_preferred_month', 'vendor_month_std', 'vendor_monthly_frequency']
Total columns: 39
Shape after dropping same-day rows: (3356, 39)
likelihood_prediction stats:
count    3356.000000
mean       95.884386
std        90.814485
min         1.000000
25%        

In [2]:
# Build a function that predicts the next purchase window for a customer+vendor pair
def predict_next_window(customer_id, vendor, df, model, features):
    """
    Given a customer and vendor, predict:
    - Which timing bucket (soon/medium/later)
    - Estimated day range for next purchase
    - So we know when to scrape for offers
    """
    bucket_ranges = {0: (1, 39), 1: (39, 110), 2: (110, 365)}
    bucket_labels = {0: "soon (1-39 days)", 1: "medium (39-110 days)", 2: "later (110+ days)"}

    # Get the customer's most recent transaction at this vendor
    mask = (df['customer_id'] == customer_id) & (df['vendor'] == vendor)
    customer_vendor_df = df[mask].sort_values('transaction_datetime')

    if customer_vendor_df.empty:
        return f"No transaction history for {customer_id} at {vendor}"

    last_row = customer_vendor_df.iloc[-1]
    X_pred = pd.DataFrame([last_row[features].fillna(0)])
    bucket = int(model.predict(X_pred)[0])
    proba = model.predict_proba(X_pred)[0]
    day_min, day_max = bucket_ranges[bucket]
    last_date = pd.to_datetime(last_row['transaction_datetime'])
    window_start = last_date + pd.Timedelta(days=day_min)
    window_end = last_date + pd.Timedelta(days=day_max)

    print(f"Customer: {customer_id[:8]}...")
    print(f"Vendor: {vendor}")
    print(f"Last purchase: {last_date.date()} (${last_row['amount']:.2f})")
    print(f"Predicted next purchase: {bucket_labels[bucket]}")
    print(f"Scrape window: {window_start.date()} → {window_end.date()}")
    print(f"Confidence: soon={proba[0]:.2f}, medium={proba[1]:.2f}, later={proba[2]:.2f}")
    return window_start, window_end

# Test on a real customer+vendor pair from the dataset
sample = df[df['vendor'].isin(['American Airlines', 'Delta Airlines', 'United Airlines'])].iloc[0]
predict_next_window(sample['customer_id'], sample['vendor'], df, best_model, features)



Customer: 025a7907...
Vendor: United Airlines
Last purchase: 2024-12-02 ($113.00)
Predicted next purchase: medium (39-110 days)
Scrape window: 2025-01-10 → 2025-03-22
Confidence: soon=0.30, medium=0.51, later=0.18


(Timestamp('2025-01-10 00:00:00'), Timestamp('2025-03-22 00:00:00'))